# Financial Independence (FI) Goals and Net Worth Visualization

## Introduction

Welcome to this tutorial on visualizing Financial Independence (FI) goals and net worth over time using Python and Matplotlib. Tracking your net worth and understanding how it aligns with your FI goals is a crucial part of personal finance management.

# Code

## Initialize environment

In [ ]:
%pip install matplotlib numpy yfinance -q

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
from typing import List, Tuple

In [ ]:
# Create your portfolio here
tickers = ['SPY','QQQ']
weights = [0.60,0.40]
portfolio = dict(zip(tickers, weights))
print('Portfolio:', portfolio)

# Define the start date and end date
start_date = '2000-01-01'
end_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
print('Start Date:', start_date)
print('End Date  :', end_date)

# Investment amounts for FI calculations
initial_investment = 10000  # Initial portfolio value in dollars
monthly_investment = 400    # Monthly contribution in dollars
print('Initial Investment:', initial_investment)
print('Monthly Investment:', monthly_investment)

# Custom annualized return for FI forecasting (optional override)
# Set to None to use historical return, or set a custom value (e.g., 0.07 for 7%)
custom_annualized_return = 0.10
print('Custom Annualized Return:', custom_annualized_return)

# Safe Withdrawal Rate (SWR) for Financial Independence calculations
# The percentage of portfolio that can be withdrawn annually in retirement
# Set to 0 to disable SWR-based FI goal calculations
safe_withdrawal_rate = 0
print('Safe Withdrawal Rate:', safe_withdrawal_rate)

## Net worth vs. time visualization

In [ ]:
def plot_net_worth(date_list: List[datetime], net_worth_list: List[float],
                   end_date: datetime = datetime.today()) -> None:
    """
    Plots the net worth over time.

    Parameters:
        date_list (List[datetime]): List of dates.
        net_worth_list (List[float]): List of net worth values corresponding to the dates in thousands ($1k).
        end_date (datetime): The end date for the x-axis limit.
    """
    # Initialize the plot with specific size and styles
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor('xkcd:light grey')
    ax.set_facecolor('xkcd:black')

    # Convert datetime objects to dates if needed
    if isinstance(date_list[0], datetime):
        date_list = [d.date() for d in date_list]

    # Net worth is already in thousands ($k)
    net_worth_k = net_worth_list

    # Plot the net worth line and fill under the line
    ax.plot(date_list, net_worth_k, linewidth=3)
    ax.fill_between(date_list, net_worth_k, alpha=0.3)

    # Highlight the last point
    ax.plot(date_list[-1], net_worth_k[-1], marker=(5, 1), markersize=10, color='orange')

    # Date formatting for x-axis
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 1)))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))

    # Set limits, labels, and title
    start_date_plot = min(date_list)
    ax.set_xlim([start_date_plot, end_date])
    ax.set_ylim([0, max(net_worth_k) * 1.1])
    ax.set_title('Net Worth', fontsize=16)
    ax.set_xlabel('Date', fontsize=16)
    ax.set_ylabel('Net Worth ($k)', fontsize=16)

    # Enable grid and set tick sizes
    ax.grid(True)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    # Show the plot
    plt.show()

# Example usage:
# plot_net_worth(date_list, net_worth_list, datetime.today())

## Visualize progress relative to financial independence (FI) goals

In [ ]:
def plot_fi_goals(net_worth_list: List[float], date_list: List[datetime], swr: float = 0.035,
                  lean_swr: float = 36, safe_swr: float = 50, cozy_swr: float = 100,
                  end_date: datetime = datetime.today(), verbose=True) -> None:
    """
    Plots the net worth over time and financial independence (FI) goals.

    Parameters:
        net_worth_list (List[float]): List of net worth values in thousands ($1k).
        date_list (List[datetime]): List of dates corresponding to the net worth values.
        swr (float): Safe withdrawal rate (default 3.5%). Set to 0 to disable SWR-based FI goals.
        lean_swr (float): Lean FI annual withdrawal amount in thousands ($1k).
        safe_swr (float): Safe FI annual withdrawal amount in thousands ($1k).
        cozy_swr (float): Cozy FI annual withdrawal amount in thousands ($1k).
        end_date (datetime): The end date for the x-axis limit.
        verbose (bool): Whether to print goal information.
    """

    # Convert datetime objects to dates if needed
    if isinstance(date_list[0], datetime):
        date_list = [d.date() for d in date_list]

    # Net worth is already in thousands ($k)
    net_worth_k = net_worth_list

    # Convert safe withdrawl rate (SWR) goals to net worth goals in thousands ($k)
    # Handle SWR = 0% edge case to avoid division by zero
    if swr > 0:
        lean_goal = lean_swr / swr
        safe_goal = safe_swr / swr
        cozy_goal = cozy_swr / swr
    else:
        # When SWR is 0%, set goals to infinity (never reachable)
        lean_goal = float('inf')
        safe_goal = float('inf')
        cozy_goal = float('inf')

    # Update matplotlib settings
    plt.rcParams.update({
        'legend.fancybox': True,
        'font.size': 16,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16
    })

    fig, ax = plt.subplots(1, 1)
    fig.patch.set_facecolor('xkcd:light grey')
    ax.set_facecolor('xkcd:black')

    # Plotting net worth
    ax.plot(date_list, net_worth_k, linewidth=3, label='Net Worth')
    ax.fill_between(date_list, net_worth_k, alpha=0.3)

    # Calculate current safe withdrawal rate (SWR)
    swr_current = np.round(swr * net_worth_list[-1], 1) if swr > 0 else 0

    if verbose:
      if swr > 0:
          print(f"Lean FI Goal ($k): {lean_goal:.1f}")
          print(f"Safe FI Goal ($k): {safe_goal:.1f}")
          print(f"Cozy FI Goal ($k): {cozy_goal:.1f}")
      else:
          print(f"SWR is 0% - FI goals disabled")
      print(f"Current NW   ($k): {net_worth_k[-1]:.1f}")

    # Horizontal lines representing different FI goals (only if SWR > 0)
    if swr > 0:
        ax.hlines(net_worth_list[-1], date_list[0], date_list[-1], colors='y', linewidth=3, label=f'SWR: ${swr_current}k/yr (current)')
        ax.hlines(lean_goal, date_list[0], date_list[-1], colors='g', linewidth=3, label=f"SWR: ${lean_swr}k/yr (lean-FI)")
        ax.hlines(safe_goal, date_list[0], date_list[-1], colors='b', linewidth=3, label=f'SWR: ${safe_swr}k/yr (safe-FI)')
        ax.hlines(cozy_goal, date_list[0], date_list[-1], colors='m', linewidth=3, label=f'SWR: ${cozy_swr}k/yr (cozy-FI)')

    # Marker at the last point
    ax.plot(date_list[-1], net_worth_k[-1], marker=(5, 1), markersize=10, color='orange')

    # Date formatting for x-axis
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 1)))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))

    # Graph settings
    start_date_plot = min(date_list)
    plt.grid(True)
    plt.xlim([start_date_plot, end_date])
    if swr > 0:
        plt.ylim([0, cozy_goal * 1.1])
    plt.title('Financial Independence (FI) Goals')
    plt.xlabel('Date')
    plt.ylabel('Net Worth ($k)')
    plt.legend(fontsize=12)
    if swr > 0:
        plt.figtext(0.5, -0.07, f'*Assumes a safe withdrawal rate (SWR) of {swr*100:.1f}%.', ha='center', fontsize=12)
    else:
        plt.figtext(0.5, -0.07, '*SWR is set to 0% - FI goal calculations disabled.', ha='center', fontsize=12)
    plt.show()

# Example usage:
# plot_fi_goals(net_worth_list, date_list, datetime.today())

## FI Goal Forecast Visualization

In [ ]:
def plot_fi_forecast(net_worth_list: List[float], date_list: List[datetime],
                     monthly_investment: float, annualized_return: float = None, swr: float = 0.035,
                     lean_swr: float = 36, safe_swr: float = 50, cozy_swr: float = 100,
                     forecast_years: int = 30, verbose=True):
    """
    Plots the net worth forecast and shows when FI goals will be reached.

    Parameters:
        net_worth_list (List[float]): List of net worth values in thousands ($1k).
        date_list (List[datetime]): List of dates corresponding to the net worth values.
        monthly_investment (float): Monthly investment amount in dollars.
        swr (float): Safe withdrawal rate (default 3.5%). Set to 0 to disable SWR-based FI goals.
        lean_swr (float): Lean FI annual withdrawal amount in thousands ($1k).
        safe_swr (float): Safe FI annual withdrawal amount in thousands ($1k).
        cozy_swr (float): Cozy FI annual withdrawal amount in thousands ($1k).
        forecast_years (int): Number of years to forecast.
        verbose (bool): Whether to print forecast information.
    
    Returns:
        Tuple: Dates when Lean, Safe, and Cozy FI goals are reached.
    """
    # Calculate FI goals in thousands ($k)
    # Handle SWR = 0% edge case to avoid division by zero
    if swr > 0:
        lean_goal = lean_swr / swr
        safe_goal = safe_swr / swr
        cozy_goal = cozy_swr / swr
    else:
        # When SWR is 0%, set goals to infinity (never reachable)
        lean_goal = float('inf')
        safe_goal = float('inf')
        cozy_goal = float('inf')
    
    # Get current net worth and date
    current_nw = net_worth_list[-1]  # in thousands
    current_date = date_list[-1]
    if isinstance(current_date, datetime):
        current_date = current_date.date()
    
    # Calculate historical annualized return
    if annualized_return is None:
            annualized_return = 0.07  # Default 7%
    
    # Generate forecast dates and values
    forecast_dates = [current_date]
    forecast_values = [current_nw]
    
    monthly_return = (1 + annualized_return) ** (1/12) - 1
    monthly_investment_k = monthly_investment / 1000  # Convert to thousands
    
    # Track when goals are reached (exact dates)
    lean_date = None
    safe_date = None
    cozy_date = None
    
    for month in range(1, forecast_years * 12 + 1):
        new_date = current_date + timedelta(days=month * 30)
        new_value = forecast_values[-1] * (1 + monthly_return) + monthly_investment_k
        
        # No withdrawals during accumulation forecast — SWR is used only
        # to compute the target NW levels (goal = annual_spend / swr).
        
        forecast_dates.append(new_date)
        forecast_values.append(new_value)
        
        # Check if goals are reached (store exact date)
        if lean_date is None and new_value >= lean_goal:
            lean_date = new_date
        if safe_date is None and new_value >= safe_goal:
            safe_date = new_date
        if cozy_date is None and new_value >= cozy_goal:
            cozy_date = new_date
    
    # Update matplotlib settings
    plt.rcParams.update({
        'legend.fancybox': True,
        'font.size': 16,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16
    })

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    fig.patch.set_facecolor('xkcd:light grey')
    ax.set_facecolor('xkcd:black')

    # Plot historical data
    hist_dates = [d.date() if isinstance(d, datetime) else d for d in date_list]
    ax.plot(hist_dates, net_worth_list, linewidth=3, label='Historical', color='cyan')
    ax.fill_between(hist_dates, net_worth_list, alpha=0.3, color='cyan')

    # Plot forecast
    ax.plot(forecast_dates, forecast_values, linewidth=3, linestyle='-', label='Forecast', color='orange', alpha=0.7)
    ax.fill_between(forecast_dates, forecast_values, alpha=0.2, color='orange')

    # Horizontal lines for FI goals (only if SWR > 0)
    if swr > 0:
        ax.hlines(lean_goal, hist_dates[0], forecast_dates[-1], colors='g', linewidth=2, label=f'Lean FI: ${lean_goal:.0f}k')
        ax.hlines(safe_goal, hist_dates[0], forecast_dates[-1], colors='b', linewidth=2, label=f'Safe FI: ${safe_goal:.0f}k')
        ax.hlines(cozy_goal, hist_dates[0], forecast_dates[-1], colors='m', linewidth=2, label=f'Cozy FI: ${cozy_goal:.0f}k')

    # Mark goal achievement at exact intersection points
    if lean_date:
        ax.axvline(lean_date, color='g', linestyle=':', alpha=0.7, linewidth=2)
    if safe_date:
        ax.axvline(safe_date, color='b', linestyle=':', alpha=0.7, linewidth=2)
    if cozy_date:
        ax.axvline(cozy_date, color='m', linestyle=':', alpha=0.7, linewidth=2)

    # Date formatting - 5 year intervals
    ax.xaxis.set_major_locator(mdates.YearLocator(base=5))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))

    # Graph settings
    plt.grid(True)
    if swr > 0:
        plt.ylim([0, cozy_goal * 1.2])
    plt.title('FI Goal Forecast', fontsize=16)
    plt.xlabel('Date', fontsize=16)
    plt.ylabel('Net Worth ($k)', fontsize=16)
    plt.legend(fontsize=10, loc='upper left')
    
    # Add forecast info text
    swr_text = f'SWR: {swr*100:.1f}%' if swr > 0 else 'SWR: 0% (disabled)'
    info_text = f'Annualized Return: {annualized_return*100:.1f}%\nMonthly Investment: ${monthly_investment:,}\n{swr_text}'
    plt.figtext(0.02, 0.02, info_text, fontsize=10, ha='left')
    
    plt.tight_layout()
    plt.show()
    
    if verbose:
        print(f"\n=== FI Goal Forecast ===")
        print(f"Current Net Worth: ${current_nw:.1f}k")
        print(f"Historical Annualized Return: {annualized_return*100:.1f}%")
        print(f"Monthly Investment: ${monthly_investment:,}")
        if swr > 0:
            print(f"Safe Withdrawal Rate: {swr*100:.1f}%")
            print(f"\nGoal Achievement Dates:")
            print(f"  Lean FI (${lean_goal:.0f}k): {lean_date.strftime('%Y-%m-%d') if lean_date else 'Not within forecast period'}")
            print(f"  Safe FI (${safe_goal:.0f}k): {safe_date.strftime('%Y-%m-%d') if safe_date else 'Not within forecast period'}")
            print(f"  Cozy FI (${cozy_goal:.0f}k): {cozy_date.strftime('%Y-%m-%d') if cozy_date else 'Not within forecast period'}")
        else:
            print(f"Safe Withdrawal Rate: 0% (SWR-based FI goals disabled)")
    
    return lean_date, safe_date, cozy_date

# Example Usage

## Load portfolio data from my_portfolio.py

In [ ]:
# Display portfolio configuration loaded from my_portfolio.py
print("Portfolio Configuration:")
print(f"Tickers: {tickers}")
print(f"Weights: {weights}")
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")
print(f"Initial Investment: ${initial_investment:,}")
print(f"Monthly Investment: ${monthly_investment:,}")

## Fetch historical price data and calculate portfolio value

In [ ]:
def fetch_portfolio_data_with_contributions(tickers, weights, start_date, end_date, 
                                            initial_investment, monthly_investment):
    """
    Fetch historical price data and calculate portfolio value over time with monthly contributions.
    
    Parameters:
        tickers (list): List of ticker symbols.
        weights (list): List of portfolio weights.
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        initial_investment (float): Initial investment amount in dollars.
        monthly_investment (float): Monthly contribution amount in dollars.
    
    Returns:
        dates (list): List of dates.
        portfolio_values (list): List of portfolio values in thousands ($1k).
    """
    # Fetch historical data for all tickers
    data = yf.download(tickers, start=start_date, end=end_date, progress=False, auto_adjust=True)
    import pandas as _pd
    if isinstance(data.columns, _pd.MultiIndex):
        data = data['Close']
    
    # Get close prices
    if len(tickers) == 1:
        prices = data.to_frame() if isinstance(data, _pd.Series) else data
        prices.columns = tickers
    else:
        prices = data
    
    # Drop rows with missing data
    prices = prices.dropna()
    
    # Resample to monthly data (use last day of each month)
    monthly_prices = prices.resample('ME').last()
    
    # Calculate monthly returns
    monthly_returns = monthly_prices.pct_change(fill_method=None).fillna(0)
    
    # Calculate portfolio value with contributions
    portfolio_values = [initial_investment]
    dates = [monthly_prices.index[0]]
    
    for i in range(1, len(monthly_prices)):
        # Apply portfolio return to current value
        portfolio_return = sum(monthly_returns.iloc[i] * weights)
        new_value = portfolio_values[-1] * (1 + portfolio_return) + monthly_investment
        portfolio_values.append(new_value)
        dates.append(monthly_prices.index[i])
    
    # Convert to thousands
    portfolio_values_k = [v / 1000 for v in portfolio_values]
    
    daily_returns = prices.pct_change(fill_method=None).dropna(how='all').fillna(0)
    weighted_daily = (daily_returns * weights).sum(axis=1)
    total_growth = (1 + weighted_daily).prod()
    years = len(weighted_daily) / 252
    annualized_return = (total_growth ** (1 / years)) - 1

    return dates, portfolio_values_k, annualized_return

## Generate charts

In [ ]:
# Fetch portfolio data using variables from my_portfolio.py
date_list, net_worth_list, annualized_return = fetch_portfolio_data_with_contributions(
    tickers=tickers,
    weights=weights,
    start_date=start_date,
    end_date=end_date,
    initial_investment=initial_investment,
    monthly_investment=monthly_investment
)

verbose = True
if verbose:
    print(f"Number of data points: {len(date_list)}")
    print(f"Date range: {date_list[0]} to {date_list[-1]}")
    print(f"Net worth range: ${net_worth_list[0]:.1f}k to ${net_worth_list[-1]:.1f}k")

### Financial Indepence (FI) Goals

The following FI goals are configurable. Adjust the values based on your personal financial goals:
- **Lean FI**: Minimum viable financial independence (lower annual expenses)
- **Safe FI**: Comfortable financial independence (moderate annual expenses)
- **Cozy FI**: Luxurious financial independence (higher annual expenses)

In [ ]:
# FI Goals - These are the only hardcoded values you should adjust
# All values are annual withdrawal amounts in thousands ($1k)

# Use safe_withdrawal_rate from my_portfolio.py (imported at the top)
# Set to 0 to disable SWR-based FI goal calculations
swr = safe_withdrawal_rate
lean_swr = 36
safe_swr = 50
cozy_swr = 100

# Plot FI goals
plot_fi_goals(net_worth_list,
              date_list,
              swr=swr,
              lean_swr=lean_swr,
              safe_swr=safe_swr,
              cozy_swr=cozy_swr,
              end_date=datetime.today())

### Plot Net Worth

In [ ]:
# Plot net worth over time
plot_net_worth(date_list, net_worth_list, end_date=datetime.today())

### FI Goal Forecast

In [ ]:
# Plot FI forecast showing when goals will be reached
lean_date, safe_date, cozy_date = plot_fi_forecast(
    net_worth_list,
    date_list,
    monthly_investment=monthly_investment,
    annualized_return=annualized_return,
    swr=swr,
    lean_swr=lean_swr,
    safe_swr=safe_swr,
    cozy_swr=cozy_swr,
    forecast_years=40
)

### FI Goal Forecast with Custom Annualized Return

This chart uses a custom annualized return rate defined in `my_portfolio.py` instead of the historical return. This allows you to model different scenarios with more conservative or optimistic return assumptions.

In [ ]:
print(f"Using custom annualized return: {custom_annualized_return*100:.1f}%")

# Plot FI forecast with custom return rate
lean_date_custom, safe_date_custom, cozy_date_custom = plot_fi_forecast(
    net_worth_list,
    date_list,
    monthly_investment=monthly_investment,
    annualized_return=custom_annualized_return,
    swr=swr,
    lean_swr=lean_swr,
    safe_swr=safe_swr,
    cozy_swr=cozy_swr,
    forecast_years=40
)

## Two-Phase Simulation: Accumulation + Withdrawal

This simulation models two distinct phases:
- **Accumulation phase**: Portfolio grows with monthly contributions until retirement
- **Withdrawal phase**: Contributions stop and a fixed annual spend is deducted

This matches the logic from the FI Forecast tab, allowing you to model scenarios for Lean, Safe, and Cozy FI spending levels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# === Two-phase simulation parameters ===
withdrawal_start_year = 20   # Years until withdrawals begin (set 0 for immediate)
total_horizon_years   = 40   # Total forecast horizon in years

# Annual spending scenarios (in $k/year)
lean_fi  = 40   # Lean FI annual spend
safe_fi  = 60   # Safe FI annual spend
cozy_fi  = 80   # Cozy FI annual spend

# Use custom_annualized_return if set, else compute from data
_, _, hist_return = fetch_portfolio_data_with_contributions(
    tickers=tickers, weights=weights, start_date=start_date, end_date=end_date,
    initial_investment=initial_investment, monthly_investment=monthly_investment
)
use_return = custom_annualized_return if custom_annualized_return and custom_annualized_return > 0 else hist_return
monthly_ret = (1 + use_return) ** (1/12) - 1

print(f'Annual return used for forecast: {use_return:.2%}')
print(f'Accumulation phase: 0 - {withdrawal_start_year} years')
print(f'Withdrawal phase:   {withdrawal_start_year} - {total_horizon_years} years')

In [ ]:
def simulate_two_phase(annual_spend_k, current_value_k, monthly_ret,
                        monthly_investment_dollars, withdrawal_start_mo, total_months):
    """
    Two-phase portfolio simulation.
    Returns list of monthly values in $k.
    """
    values = [current_value_k]
    monthly_withdrawal = annual_spend_k / 12
    for mo in range(total_months):
        prev = max(values[-1], 0)
        grown = prev * (1 + monthly_ret)
        if mo < withdrawal_start_mo:
            next_val = grown + monthly_investment_dollars / 1000
        else:
            next_val = grown - monthly_withdrawal
        values.append(next_val)
    return values

# Current portfolio value (last value from historical data)
_, net_worth_k, _ = fetch_portfolio_data_with_contributions(
    tickers=tickers, weights=weights, start_date=start_date, end_date=end_date,
    initial_investment=initial_investment, monthly_investment=monthly_investment
)
current_value_k = net_worth_k[-1]
total_months = total_horizon_years * 12
withdrawal_start_mo = withdrawal_start_year * 12
proj_dates = pd.date_range(pd.Timestamp.today(), periods=total_months + 1, freq='MS')

scenarios = [
    (lean_fi, 'Lean FI',  'steelblue'),
    (safe_fi, 'Safe FI',  'limegreen'),
    (cozy_fi, 'Cozy FI',  'orange'),
]

fig, ax = plt.subplots(figsize=(14, 6))

# Shade phases
acc_end_date = proj_dates[withdrawal_start_mo]
ax.axvspan(proj_dates[0], acc_end_date, alpha=0.06, color='green', label='_nolegend_')
ax.axvspan(acc_end_date, proj_dates[-1], alpha=0.06, color='red',   label='_nolegend_')
ax.axvline(acc_end_date, color='white', linewidth=1.5, linestyle=':', alpha=0.7)
ax.text(acc_end_date, ax.get_ylim()[1] if ax.get_ylim()[1] != 1 else 1,
        f' Retirement yr {withdrawal_start_year}', color='gray', fontsize=9)

for spend_k, label, color in scenarios:
    vals = simulate_two_phase(
        spend_k, current_value_k, monthly_ret,
        monthly_investment, withdrawal_start_mo, total_months
    )
    # Accumulation solid, withdrawal dashed
    ax.plot(proj_dates[:withdrawal_start_mo+1], vals[:withdrawal_start_mo+1],
            color=color, linewidth=2)
    ax.plot(proj_dates[withdrawal_start_mo:], vals[withdrawal_start_mo:],
            color=color, linewidth=2, linestyle='--',
            label=f'{label} (${spend_k}k/yr spend)')
    # Mark if portfolio exhausted
    exhausted = [i for i, v in enumerate(vals[withdrawal_start_mo:]) if v <= 0]
    if exhausted:
        ex_mo = withdrawal_start_mo + exhausted[0]
        ax.scatter([proj_dates[ex_mo]], [0], color=color, s=80, zorder=6, marker='X')

ax.axhline(0, color='gray', linewidth=0.7, alpha=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x*1000:,.0f}' if abs(x) < 1000 else f'${x/1000:.1f}M'
))
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value')
ax.set_title(
    f'Two-Phase FI Forecast  |  {use_return:.1%} return  |  '
    f'${monthly_investment:,}/mo contributions  |  '
    f'Retirement at year {withdrawal_start_year}'
)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Monte Carlo Simulation with Accumulation & Withdrawal Phases

Runs thousands of random return paths using the portfolio's historical volatility, allowing you to see the distribution of outcomes including the probability of portfolio exhaustion.

In [ ]:
# MC parameters
n_simulations   = 5000
mc_spend_k      = safe_fi  # Which spending scenario to simulate
n_paths_display = 100      # How many individual paths to show

# Historical portfolio daily returns
import yfinance as yf
raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True, progress=False)
import pandas as pd
if isinstance(raw_data.columns, pd.MultiIndex):
    raw_data = raw_data['Close']
daily_ret = raw_data.pct_change(fill_method=None).dropna(how='all').fillna(0)
p_daily_ret = daily_ret.dot(weights)

# Monthly statistics for MC
monthly_vol = p_daily_ret.std() * (252 ** 0.5) / (12 ** 0.5)  # monthly sigma

rng = np.random.default_rng(42)
mc_paths = []
for _ in range(n_simulations):
    vals = [current_value_k]
    for mo in range(total_months):
        r = rng.normal(monthly_ret, monthly_vol)
        grown = max(vals[-1], 0) * (1 + r)
        if mo < withdrawal_start_mo:
            next_val = grown + monthly_investment / 1000
        else:
            next_val = grown - mc_spend_k / 12
        vals.append(next_val)
    mc_paths.append(vals)

mc_arr = np.array(mc_paths)
p10 = np.percentile(mc_arr, 10, axis=0)
p50 = np.percentile(mc_arr, 50, axis=0)
p90 = np.percentile(mc_arr, 90, axis=0)
surviving = np.sum(mc_arr[:, -1] > 0) / n_simulations * 100

fig, ax = plt.subplots(figsize=(14, 6))

# Shade phases
ax.axvspan(proj_dates[0], acc_end_date, alpha=0.05, color='green')
ax.axvspan(acc_end_date, proj_dates[-1], alpha=0.05, color='red')
ax.axvline(acc_end_date, color='gray', linewidth=1, linestyle=':')

# Individual paths
for path in mc_paths[:n_paths_display]:
    ax.plot(proj_dates, path, alpha=0.04, color='steelblue', linewidth=0.7)

ax.plot(proj_dates, p50, color='steelblue', linewidth=2, label='Median (50th pct)')
ax.plot(proj_dates, p10, color='red',       linewidth=1.5, linestyle='--', label='Pessimistic (10th pct)')
ax.plot(proj_dates, p90, color='limegreen', linewidth=1.5, linestyle='--', label='Optimistic (90th pct)')
ax.fill_between(proj_dates, p10, p90, alpha=0.08, color='steelblue')
ax.axhline(0, color='gray', linewidth=0.7, alpha=0.5)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x*1000:,.0f}' if abs(x) < 1000 else f'${x/1000:.1f}M'
))
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value')
ax.set_title(
    f'Monte Carlo: {n_simulations} paths  |  Safe FI ${mc_spend_k}k/yr spend  |  '
    f'Retirement yr {withdrawal_start_year}  |  '
    f'Portfolio survives in {surviving:.0f}% of scenarios'
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'At year {total_horizon_years}:')
print(f'  Median:        ${p50[-1]*1000:,.0f}')
print(f'  10th pct:      ${p10[-1]*1000:,.0f}')
print(f'  90th pct:      ${p90[-1]*1000:,.0f}')
print(f'  Survived ({surviving:.0f}%): {int(surviving/100*n_simulations)}/{n_simulations} paths')